<div align="center">
  <h1>Medical RAG Assistant</h1>
  <h3>Conversational medical Q&A with RAG, ChromaDB, and LangChain ReAct Agent</h3>
  <p>by <b><a href="https://www.linkedin.com/in/adebanji-adelowo/">Adebanji Oluwatimileyin Adelowo</a></b> &nbsp;|&nbsp; <a href="https://github.com/AdebanjiAdelowo">GitHub</a></p>
</div>

---

**Models:** OpenAI GPT + text-embedding-ada-002  
**Environment:** Google Colab (CPU) or local  
**Tags:** RAG, ChromaDB, LangChain, Embeddings, OpenAI, ReAct Agent

---

#Installing libraries & Loading Dataset

In [ ]:
!pip install - langchain langchain-classic langchain-openai langchain-community langchain-chroma langchain-text-splitters datasets chromadb openai langgraph

We will download the dataset from the Hugging Face datasets library. It's a dataset with information about diseases.

In [60]:
from datasets import load_dataset

data = load_dataset("keivalya/MedQuad-MedicalQnADataset", split='train')


In [61]:
data = data.to_pandas()
data.head(10)

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."
5,prevention,How to prevent Lymphocytic Choriomeningitis (L...,LCMV infection can be prevented by avoiding co...
6,information,What is (are) Parasites - Cysticercosis ?,Cysticercosis is an infection caused by the la...
7,susceptibility,Who is at risk for Parasites - Cysticercosis? ?,Cysticercosis is an infection caused by the la...
8,exams and tests,How to diagnose Parasites - Cysticercosis ?,"If you think that you may have cysticercosis, ..."
9,treatment,What are the treatments for Parasites - Cystic...,Some people with cysticercosis do not need to ...


In [62]:
#uncoment this line if you want to limit the size of the data.
data = data[0:100]

As you can see, the medical information in the dataset is well-organized, and to someone like me, who is not an expert in the field, it appears to be quite valuable. This information could be a useful addition to any general medicine book to support primary care doctors.

Load the langchain libraries to load the document.

In [63]:
from langchain_community.document_loaders import DataFrameLoader
from langchain_chroma import Chroma

The Document is in the Answer column, and the others columns are Metadata.

In [64]:
df_loader = DataFrameLoader(data, page_content_column="Answer")


In [65]:
df_document = df_loader.load()
display(df_document[:2])

[Document(metadata={'qtype': 'susceptibility', 'Question': 'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?'}, page_content='LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.'),
 Document(metadata={'qtype': 'symptoms', 'Question': 'What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?'}, page_content='LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symptoms or mild febrile illnesses are more common clinical manifestations. \n                \nFor infected persons who do become ill, onset of sympt

We can chunk the documents. The size to which we want to split the document is a design decision. The larger it is, the larger the prompt will be, and the slower the Model's response process.

We also need to consider the maximum prompt size and ensure that the document does not exceed it.

In [66]:
from langchain_text_splitters import CharacterTextSplitter

In [67]:
text_splitter = CharacterTextSplitter(chunk_size=1250,
                                      separator="\n",
                                      chunk_overlap=100)
texts = text_splitter.split_documents(df_document)


These warnings we see are because it can't perform the partition of the required size. This is because it waits for a page break to divide the text and does so when possible.

In [68]:
first_doc = texts[1]
print(first_doc.page_content)

LCMV is most commonly recognized as causing neurological disease, as its name implies, though infection without symptoms or mild febrile illnesses are more common clinical manifestations. 
                
For infected persons who do become ill, onset of symptoms usually occurs 8-13 days after exposure to the virus as part of a biphasic febrile illness. This initial phase, which may last as long as a week, typically begins with any or all of the following symptoms: fever, malaise, lack of appetite, muscle aches, headache, nausea, and vomiting. Other symptoms appearing less frequently include sore throat, cough, joint pain, chest pain, testicular pain, and parotid (salivary gland) pain.


### Initialize the Embedding Model and Vector DB

We load the text-embedding-ada-002 model from OpenAI.

In [69]:
from getpass import getpass
OPENAI_API_KEY = getpass("OpenAI API Key: ")

In [70]:
from langchain_openai import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=OPENAI_API_KEY
)

The execution of this cell may take 3 to 5 minutes. If you want it to be faster, you can reduce the number of records in the dataset.

In [71]:
import shutil, os

directory_cdb = '/content/drive/MyDrive/chromadb2'

chroma_db = Chroma.from_documents(
    df_document, embed, persist_directory=directory_cdb
)

We are going to create three objects.

* The language model, which can be any of those from OpenAI.
* The memory, responsible for keeping the prompt with all the necessary history.
* The retrieval, used to obtain information stored in ChromaDB.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# =============================================================================
# 1. LLM SETUP
# =============================================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=OPENAI_API_KEY,
    temperature=0
)

system_prompt = (
    "You are a helpful medical assistant for question-answering tasks. "
    "Use the following retrieved context to answer the question accurately. "
    "If you don't know the answer, say that you don't know. "
    "Keep your answer concise and factual.\n\n"
    "{context}"
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [81]:
# Combines retrieved documents into a single context block for the LLM
question_answer_chain = create_stuff_documents_chain(llm, rag_prompt)

# Handles retrieval from Chroma and passes docs to the chain above
qa_chain = create_retrieval_chain(
    chroma_db.as_retriever(),
    question_answer_chain
)

We can try the isolated Retrieval to see if the information it returns is relevant.




In [82]:
# Test standalone
response = qa_chain.invoke({"input": "What is the main symptom of LCM?"})
print(response["answer"])

The main symptom of LCM (Lymphocytic Choriomeningitis) is typically a biphasic febrile illness that begins with non-specific flu-like symptoms, including fever, malaise, lack of appetite, muscle aches, headache, nausea, and vomiting. In some cases, neurological symptoms may develop in the second phase of the illness.


Perfect! The information returned is exactly what we desired.

## Creating the Agent.

In [83]:
from langchain_core.tools import tool

# =============================================================================
# 3. TOOL DEFINITION
# =============================================================================

@tool
def medical_kb(query: str) -> str:
    """
    Use this tool when answering medical knowledge queries.
    It searches a medical knowledge base and returns relevant information
    about diseases, symptoms, diagnoses, and treatments.
    """
    return qa_chain.invoke({"input": query})["answer"]

tools = [medical_kb]

In [ ]:
# =============================================================================
# 4. MEMORY SETUP
# =============================================================================
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [85]:
# =============================================================================
# 5. AGENT SETUP

# =============================================================================
from langchain.agents import create_agent 

agent_executor = create_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
    system_prompt=(                               # ✅ renamed from prompt= to system_prompt=
        "You are a helpful and knowledgeable assistant. "
        "For general questions, answer directly. "
        "For medical knowledge questions, always use the medical_kb tool "
        "to retrieve accurate, evidence-based information."
    )
)

In [86]:
# =============================================================================
# 6. RUNNING THE AGENT
# =============================================================================

def get_response(result: dict) -> str:
    return result["messages"][-1].content

### Using the Conversational Agent

To make queries we simply call the `agent` directly.

First i will try a order not related to the Medical field.

In [87]:
# --- Session 1 ---
config_session_1 = {"configurable": {"thread_id": "session-1"}}

In [88]:
result1 = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Give me the area of a square of 2x2"}]},
    config=config_session_1
)
print(get_response(result1))

The area of a square is calculated using the formula:

\[
\text{Area} = \text{side} \times \text{side}
\]

For a square with a side length of 2 units:

\[
\text{Area} = 2 \times 2 = 4 \text{ square units}
\]

So, the area of a square with side length 2 is 4 square units.


Perfect, the model has responded without accessing the configured knowledge database.

Now I will try with a question that is also not related to health.

In [89]:
result2 = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Do you know who is Clark Kent?"}]},
    config=config_session_1
)
print(get_response(result2))

Yes, Clark Kent is a fictional character and the alter ego of Superman, a superhero in comic books published by DC Comics. Created by writer Jerry Siegel and artist Joe Shuster, Clark Kent is depicted as a mild-mannered reporter for the Daily Planet in Metropolis. He is known for his iconic glasses and his contrasting persona as Superman, who possesses superhuman abilities and fights for justice. The character has been featured in various media, including comic books, television shows, and movies.


It has not accessed either, as the model has been able to identify that it is not a question related to the database that LangChain provides.

Now it's time to try with a question related to Medicine. Let's see if the model can understand that it should first look for information in the vector database at its disposal.

In [91]:
# --- New Session (replaces memory.clear()) ---
config_session_2 = {"configurable": {"thread_id": "session-2"}}

In [92]:
result3 = agent_executor.invoke(
    {"messages": [{"role": "user", "content": (
        "I have a patient that can have Botulism, "
        "how can I confirm the diagnosis?"
    )}]},
    config=config_session_2
)
print(get_response(result3))

To confirm a diagnosis of botulism, the following steps are typically taken:

1. **Detection of Botulinum Toxin**: Special tests are conducted to detect the presence of botulinum toxin in the patient's serum, stool, or food samples. These tests are available at some state health department laboratories and the CDC.

2. **Isolation of Clostridium botulinum**: Culturing the bacteria from the patient's stool or wound can also help confirm the diagnosis.

3. **Exclusion of Other Conditions**: Since botulism symptoms can mimic other neurological conditions, tests may be performed to rule out other diagnoses such as:
   - Guillain-Barré syndrome
   - Stroke
   - Myasthenia gravis

   These tests may include:
   - Brain imaging (CT or MRI)
   - Spinal fluid examination
   - Nerve conduction studies (electromyography, or EMG)
   - Tensilon test for myasthenia gravis

It's important to act quickly, as botulism can be life-threatening. If you suspect botulism, consider initiating treatment while

Perfect, the most important thing for us is that it has been able to identify that it should go to the medical database to search for information about the symptoms.

In [93]:
result3 = agent_executor.invoke(
    {"messages": [{"role": "user", "content": (
        "Is this an important illness?"
    )}]},
    config=config_session_2
)
print(get_response(result3))

Yes, botulism is an important illness to recognize and understand due to its potential severity and the risk of fatality. Here are some key points regarding its significance:

1. **Seriousness**: Botulism is a rare but serious paralytic illness caused by a potent nerve toxin produced by the bacterium *Clostridium botulinum*. It can lead to severe complications and can be life-threatening.

2. **Public Health Concern**: Foodborne botulism can result in outbreaks that affect multiple individuals, making it a significant public health issue.

3. **Medical Emergency**: All forms of botulism (foodborne, wound, and infant botulism) are considered medical emergencies that require immediate treatment.

4. **Preventable**: Understanding the causes and prevention methods, such as safe food handling and proper canning practices, can help reduce the incidence of botulism.

5. **Awareness of Risk Factors**: Certain behaviors, such as injecting black-tar heroin, can increase the risk of wound botuli

And the memory works perfectly. We can maintain a conversation, taking into account that the model knows the previous questions and answers.

# Conclusions.
The experiment has been a small success. The Vectorial database has been configured and filled with information from the dataset. A LangChain agent has been created, and it has been able to retrieve information from the database only when necessary. Don't forget that our ChatBot has memory.

All of this in just a few lines of code!


---